# WIFUSDC：全面查看三类原始数据

默认读取2026-10-01 01:00–02:00 UTC小时。所有表保留原字段、原始值和列顺序，不换算时间、不添加派生字段。价格/数量仍为字符串，时间/编号仍为整数，缺失值显示为`<NA>`（原始null）。

盘口按**快照买盘、快照卖盘、增量买盘、增量卖盘**分别展示；三类文件还展示开头、中间、结尾记录。每张表可横向滚动查看完整字段。不会一次输出数万行：各表均标明抽取位置，可通过最后的分页函数查看任意行段。

received_time单位纳秒；event_time、transaction_time、trade_time、next_funding_time单位毫秒。Parquet是原始二进制格式，下方JSON只是同一原始记录的可读表示。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'data/cryptohftdata/raw').is_dir())
DATE, HOUR = '2026-10-01', '01'
DATA_DIR = PROJECT_ROOT / 'data/cryptohftdata/raw/binance_futures' / DATE / HOUR
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 220)

import pyarrow.compute as pc
tables = {kind: pq.read_table(DATA_DIR / f'WIFUSDC_{kind}.parquet')
          for kind in ['orderbook', 'trades', 'mark_price']}
def show_rows(table, start=0, count=5):
    print(f'原文件/当前筛选结果的行位置：{start}至{min(start + count, table.num_rows) - 1}，从0计数')
    display(table.slice(start, count).to_pandas(types_mapper=pd.ArrowDtype))


## 文件概览

In [ ]:
display(pd.DataFrame([{'文件': f'WIFUSDC_{k}.parquet', '行数': t.num_rows, '列数': t.num_columns}
                      for k, t in tables.items()]))

## 1. orderbook：买卖两侧与两种记录

一行是一个价位，不能把前几行当作完整盘口。`snapshot`是初始化快照，`update`是价位增量；bid为买盘，ask为卖盘。quantity为该档最新总量，0表示删除。

下面四组均保持文件原顺序，**没有按价格重新排序**；“前5行”不一定是最优五档。

In [ ]:
book = tables['orderbook']
print(book.schema)
display(book.select(['event_type', 'side']).to_pandas().value_counts(sort=False).rename('行数').reset_index())

### snapshot / bid：完整原始字段

In [ ]:
selected = book.filter(pc.and_(pc.equal(book['event_type'], 'snapshot'), pc.equal(book['side'], 'bid')))
print('该类记录数：', selected.num_rows)
show_rows(selected)
if selected.num_rows:
    print('第一行原始记录：')
    print(json.dumps(selected.slice(0, 1).to_pylist()[0], ensure_ascii=False, indent=2))

### snapshot / ask：完整原始字段

In [ ]:
selected = book.filter(pc.and_(pc.equal(book['event_type'], 'snapshot'), pc.equal(book['side'], 'ask')))
print('该类记录数：', selected.num_rows)
show_rows(selected)
if selected.num_rows:
    print('第一行原始记录：')
    print(json.dumps(selected.slice(0, 1).to_pylist()[0], ensure_ascii=False, indent=2))

### update / bid：完整原始字段

In [ ]:
selected = book.filter(pc.and_(pc.equal(book['event_type'], 'update'), pc.equal(book['side'], 'bid')))
print('该类记录数：', selected.num_rows)
show_rows(selected)
if selected.num_rows:
    print('第一行原始记录：')
    print(json.dumps(selected.slice(0, 1).to_pylist()[0], ensure_ascii=False, indent=2))

### update / ask：完整原始字段

In [ ]:
selected = book.filter(pc.and_(pc.equal(book['event_type'], 'update'), pc.equal(book['side'], 'ask')))
print('该类记录数：', selected.num_rows)
show_rows(selected)
if selected.num_rows:
    print('第一行原始记录：')
    print(json.dumps(selected.slice(0, 1).to_pylist()[0], ensure_ascii=False, indent=2))

### 同一盘口更新消息中的买卖盘
一条消息可能只有买盘、只有卖盘，也可能同时包含两侧。以下在本小时中寻找同时含bid/ask的相邻消息组，展示该组两侧；仅作原始消息结构浏览。

In [ ]:
rows = book.to_pylist()
groups = []
current, previous = [], None
for row in rows:
    if row['event_type'] != 'update':
        continue
    key = (row['received_time'], row['event_time'], row['final_update_id'])
    if key != previous and current:
        groups.append(current)
        current = []
    current.append(row)
    previous = key
if current:
    groups.append(current)
mixed = next((g for g in groups if {r['side'] for r in g} == {'bid', 'ask'}), None)
if mixed:
    print('该消息组价位行数：', len(mixed), '；final_update_id：', mixed[0]['final_update_id'])
    for side in ['bid', 'ask']:
        print(side)
        preview = [r for r in mixed if r['side'] == side][:5]
        import pyarrow as pa
        display(pa.Table.from_pylist(preview, schema=book.schema).to_pandas(types_mapper=pd.ArrowDtype))
else:
    print('本小时未找到同时含两侧的消息组；这不表示文件没有卖盘。')

## 盘口文件整体：开头、中间、结尾

快照通常在起点，后续是增量；中间/结尾仍保留完整更新编号。

In [ ]:
table = tables['orderbook']
print(table.schema)

### 开头5行

In [ ]:
show_rows(table, start=0, count=5)

### 中间5行

In [ ]:
show_rows(table, start=max(0, table.num_rows // 2 - 2), count=5)

### 结尾5行

In [ ]:
show_rows(table, start=max(0, table.num_rows - 5), count=5)

## 逐笔成交：开头、中间、结尾

每行是一筆真实成交。is_buyer_maker=true表示主动卖出，false表示主动买入；quantity是成交量，不是挂单量。

In [ ]:
table = tables['trades']
print(table.schema)

### 开头5行

In [ ]:
show_rows(table, start=0, count=5)

### 中间5行

In [ ]:
show_rows(table, start=max(0, table.num_rows // 2 - 2), count=5)

### 结尾5行

In [ ]:
show_rows(table, start=max(0, table.num_rows - 5), count=5)

In [ ]:
print('第一行完整原始记录：')
print(json.dumps(table.slice(0, 1).to_pylist()[0], ensure_ascii=False, indent=2))

## 标记价：开头、中间、结尾

每行是一条估值行情。mark_price为标记价，index_price为指数价；funding_rate是原始小数费率，next_funding_time是下次资金费时间。它不是可以直接成交的买卖报价。

In [ ]:
table = tables['mark_price']
print(table.schema)

### 开头5行

In [ ]:
show_rows(table, start=0, count=5)

### 中间5行

In [ ]:
show_rows(table, start=max(0, table.num_rows // 2 - 2), count=5)

### 结尾5行

In [ ]:
show_rows(table, start=max(0, table.num_rows - 5), count=5)

In [ ]:
print('第一行完整原始记录：')
print(json.dumps(table.slice(0, 1).to_pylist()[0], ensure_ascii=False, indent=2))

## 查看任意行段

修改下面的类型、起始行和数量即可分页查看。start从0计数；kind可选orderbook、trades、mark_price。可以改成count=20或100，不建议一次展示整个文件。输出仍为原始字段。

本笔记本展示的是一个小时的真实样本，不代表所有小时具有完全相同的记录数量或快照布局。

In [ ]:
kind, start, count = 'orderbook', 900, 10
show_rows(tables[kind], start=start, count=count)